# L2.01: Automation with KFP for Gemini Tuning

- You will use [Kubeflow Pipelines](https://www.kubeflow.org/docs/components/pipelines/v2/) to orchestrate a Gemini tuning workflow.
- This version seeds the local JSONL files to GCS once, then uses KFP components to stage run-specific datasets, launch Gemini tuning, and wait for terminal success or failure.

In [1]:
from kfp import dsl
from kfp import compiler

# Ignore FutureWarnings in kfp
import warnings
warnings.filterwarnings("ignore", 
                        category=FutureWarning, 
                        module='kfp.*')

<!-- ## Kubeflow Pipelines

- Kubeflow pipelines consist of two key concepts: Components and pipelines.
- Pipeline components are like self-contained sets of code that perform various steps in your ML workflow, such as, the first step could be preprocessing data, and second step could betraining a model.

### Simple Pipeline Example 

##### Build the pipeline -->

In [2]:
### these are the same 
### jsonl files from the previous lab

### time stamps have been removed so that 
### the files are consistent for all learners
TRAINING_DATA_URI = "./tune_data_stack_overflow_gemini_python_qa-pilot-500.jsonl"
EVALUATION_DATA_URI = "./tune_eval_data_stack_overflow_gemini_python_qa-pilot-100.jsonl"


<!-- - Provide the model with a version.
- Versioning model allows for:
  - Reproducibility: Reproduce your results and ensure your models perform as expected.
  - Auditing: Track changes to your models.
  - Rollbacks: Roll back to a previous version of your model. -->

In [3]:
PIPELINE_PACKAGE_PATH = "gemini_tuning_pipeline.json"

In [4]:
import datetime

In [5]:
date = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [6]:
MODEL_NAME = f"deep-learning-ai-model-01-{date}"

- This example keeps two tuning parameters for demonstration:
  - `TRAINING_STEPS`: Number of training steps to use when tuning the model. For extractive QA you can set it from 100-500. 
  - `EVALUATION_INTERVAL`: The interval determines how frequently a trained model is evaluated against the created *evaluation set* to assess its performance and identify issues. Default will be 20, which means after every 20 training steps, the model is evaluated on the evaluation dataset.

In [7]:
TRAINING_STEPS = 200
EVALUATION_INTERVAL = 20

- Load the Project ID and credentials

In [8]:
from utils import authenticate
credentials, PROJECT_ID = authenticate() 

In [9]:
REGION = "us-central1"

- Define the arguments used by the KFP pipeline and Gemini tuning step.

In [10]:
pipeline_arguments = {
    "project_id": PROJECT_ID,
    "region": REGION,
    "model_display_name": MODEL_NAME,
    "source_model": "gemini-2.5-flash-lite",
    "poll_interval_seconds": 60,
}

In [11]:
pipeline_root = "gs://first-llmops-demo-bucket-2026/pipeline-root"
print("This bucket stores uploaded datasets and Vertex AI Pipeline artifacts.")

This bucket stores uploaded datasets and Vertex AI Pipeline artifacts.


In [12]:
import os
from google.cloud import storage

if not pipeline_root.startswith("gs://"):
    raise ValueError("Set pipeline_root to a real gs:// bucket path before submitting tuning.")

bucket_and_prefix = pipeline_root.removeprefix("gs://")
bucket_name, _, prefix = bucket_and_prefix.partition("/")
prefix = prefix.strip("/")

storage_client = storage.Client(project=PROJECT_ID, credentials=credentials)
bucket = storage_client.bucket(bucket_name)

def upload_local_seed_data(local_path, destination_name):
    blob_name = f"{prefix}/{destination_name}" if prefix else destination_name
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(local_path)
    return f"gs://{bucket_name}/{blob_name}"

training_source_gcs_uri = upload_local_seed_data(
    TRAINING_DATA_URI,
    f"source-data/training/{os.path.basename(TRAINING_DATA_URI)}",
)
evaluation_source_gcs_uri = upload_local_seed_data(
    EVALUATION_DATA_URI,
    f"source-data/validation/{os.path.basename(EVALUATION_DATA_URI)}",
)

print(training_source_gcs_uri)
print(evaluation_source_gcs_uri)

gs://first-llmops-demo-bucket-2026/pipeline-root/source-data/training/tune_data_stack_overflow_gemini_python_qa-pilot-500.jsonl
gs://first-llmops-demo-bucket-2026/pipeline-root/source-data/validation/tune_eval_data_stack_overflow_gemini_python_qa-pilot-100.jsonl


In [13]:
@dsl.component(
    base_image="python:3.11",
    packages_to_install=["google-cloud-storage>=2.19.0"],
)
def stage_dataset_for_tuning(
    source_gcs_uri: str,
    pipeline_root: str,
    model_display_name: str,
    dataset_split: str,
) -> str:
    from google.cloud import storage

    def split_gcs_uri(gcs_uri: str) -> tuple[str, str]:
        if not gcs_uri.startswith("gs://"):
            raise ValueError(f"Expected gs:// URI, got: {gcs_uri}")
        bucket_and_path = gcs_uri.removeprefix("gs://")
        bucket_name, _, blob_name = bucket_and_path.partition("/")
        if not bucket_name or not blob_name:
            raise ValueError(f"Incomplete GCS URI: {gcs_uri}")
        return bucket_name, blob_name

    destination_bucket_name, root_prefix = split_gcs_uri(pipeline_root)
    source_bucket_name, source_blob_name = split_gcs_uri(source_gcs_uri)
    root_prefix = root_prefix.rstrip("/")
    destination_blob_name = "/".join(
        part
        for part in [root_prefix, "staged-datasets", model_display_name, f"{dataset_split}.jsonl"]
        if part
    )

    storage_client = storage.Client()
    source_bucket = storage_client.bucket(source_bucket_name)
    destination_bucket = storage_client.bucket(destination_bucket_name)
    source_blob = source_bucket.blob(source_blob_name)
    destination_blob = destination_bucket.blob(destination_blob_name)

    print(f"Preparing to stage dataset_split={dataset_split}")
    print(f"Source URI: {source_gcs_uri}")
    print(f"Destination URI: gs://{destination_bucket_name}/{destination_blob_name}")

    if not source_blob.exists(storage_client):
        raise FileNotFoundError(f"Source dataset does not exist: {source_gcs_uri}")

    destination_bucket.reload(client=storage_client)
    print(f"Destination bucket confirmed: {destination_bucket.name}")

    rewrite_token = None
    while True:
        rewrite_token, bytes_rewritten, total_bytes = destination_blob.rewrite(
            source=source_blob,
            token=rewrite_token,
            client=storage_client,
        )
        print(f"Rewrite progress: {bytes_rewritten}/{total_bytes} bytes")
        if not rewrite_token:
            break

    staged_uri = f"gs://{destination_bucket_name}/{destination_blob_name}"
    print(f"Staged {source_gcs_uri} to {staged_uri}")
    return staged_uri


@dsl.component(
    base_image="python:3.11",
    packages_to_install=["google-cloud-aiplatform>=1.111.0"],
)
def start_gemini_tuning(
    project_id: str,
    region: str,
    model_display_name: str,
    source_model: str,
    train_dataset_gcs_uri: str,
    evaluation_dataset_gcs_uri: str,
    pipeline_root: str,
) -> str:
    import vertexai
    from vertexai.tuning import sft

    vertexai.init(project=project_id, location=region)
    output_uri = f"{pipeline_root.rstrip('/')}/tuning-artifacts/{model_display_name}"
    tuning_job = sft.train(
        source_model=source_model,
        train_dataset=train_dataset_gcs_uri,
        validation_dataset=evaluation_dataset_gcs_uri,
        tuned_model_display_name=model_display_name,
        output_uri=output_uri,
    )
    print(tuning_job.resource_name)
    return tuning_job.resource_name


@dsl.component(
    base_image="python:3.11",
    packages_to_install=["google-cloud-aiplatform>=1.111.0"],
)
def wait_for_gemini_tuning(
    project_id: str,
    region: str,
    tuning_job_name: str,
    poll_interval_seconds: int = 60,
) -> str:
    import time
    import vertexai
    from vertexai.tuning import sft

    vertexai.init(project=project_id, location=region)
    tuning_job = sft.SupervisedTuningJob(tuning_job_name=tuning_job_name)

    while True:
        tuning_job.refresh()
        state_name = getattr(tuning_job.state, "name", str(tuning_job.state))
        print(f"Tuning job {tuning_job_name} state: {state_name}")
        if tuning_job.has_ended:
            break
        time.sleep(max(poll_interval_seconds, 30))

    if state_name != "JOB_STATE_SUCCEEDED":
        raise RuntimeError(f"Tuning job {tuning_job_name} ended with state {state_name}")

    tuned_model_name = tuning_job.tuned_model_name or ""
    print(f"Tuned model: {tuned_model_name}")
    return tuned_model_name


In [14]:
@dsl.pipeline(name="gemini-tuning-pipeline")
def gemini_tuning_pipeline(
    project_id: str,
    region: str,
    model_display_name: str,
    source_model: str,
    pipeline_root: str,
    training_source_gcs_uri: str,
    evaluation_source_gcs_uri: str,
    poll_interval_seconds: int = 60,
):
    train_stage_task = stage_dataset_for_tuning(
        source_gcs_uri=training_source_gcs_uri,
        pipeline_root=pipeline_root,
        model_display_name=model_display_name,
        dataset_split="training",
    )
    train_stage_task.set_caching_options(False)
    train_stage_task.set_cpu_limit("1")
    train_stage_task.set_memory_limit("4Gi")

    eval_stage_task = stage_dataset_for_tuning(
        source_gcs_uri=evaluation_source_gcs_uri,
        pipeline_root=pipeline_root,
        model_display_name=model_display_name,
        dataset_split="validation",
    )
    eval_stage_task.after(train_stage_task)
    eval_stage_task.set_caching_options(False)
    eval_stage_task.set_cpu_limit("1")
    eval_stage_task.set_memory_limit("4Gi")

    start_tuning_task = start_gemini_tuning(
        project_id=project_id,
        region=region,
        model_display_name=model_display_name,
        source_model=source_model,
        train_dataset_gcs_uri=train_stage_task.output,
        evaluation_dataset_gcs_uri=eval_stage_task.output,
        pipeline_root=pipeline_root,
    )
    start_tuning_task.set_caching_options(False)
    start_tuning_task.set_cpu_limit("1")
    start_tuning_task.set_memory_limit("4Gi")

    wait_task = wait_for_gemini_tuning(
        project_id=project_id,
        region=region,
        tuning_job_name=start_tuning_task.output,
        poll_interval_seconds=poll_interval_seconds,
    )
    wait_task.set_caching_options(False)
    wait_task.set_cpu_limit("1")
    wait_task.set_memory_limit("4Gi")


In [15]:
compiler.Compiler().compile(
    pipeline_func=gemini_tuning_pipeline,
    package_path=PIPELINE_PACKAGE_PATH,
)
print(PIPELINE_PACKAGE_PATH)


gemini_tuning_pipeline.json


In [16]:
from google.cloud.aiplatform import PipelineJob

DEFAULT_PIPELINE_RUNNER_SERVICE_ACCOUNT = f"vertex-pipeline-runner@{PROJECT_ID}.iam.gserviceaccount.com"

PIPELINE_RUNNER_SERVICE_ACCOUNT = (
    globals().get("PIPELINE_RUNNER_SERVICE_ACCOUNT")
    or globals().get("PIPELINE_SERVICE_ACCOUNT")
    or os.getenv("VERTEX_PIPELINE_SERVICE_ACCOUNT")
    or DEFAULT_PIPELINE_RUNNER_SERVICE_ACCOUNT
    or getattr(credentials, "service_account_email", None)
)

if not PIPELINE_RUNNER_SERVICE_ACCOUNT:
    raise ValueError(
        "Set PIPELINE_RUNNER_SERVICE_ACCOUNT in a notebook cell, for example: "
        "PIPELINE_RUNNER_SERVICE_ACCOUNT = 'your-runner-sa@your-project.iam.gserviceaccount.com'. "
        "You can also set VERTEX_PIPELINE_SERVICE_ACCOUNT as an environment variable. "
        "Vertex otherwise falls back to the default Compute Engine service account, which caused the permission error."
    )

print(f"Submitting pipeline as: {PIPELINE_RUNNER_SERVICE_ACCOUNT}")

vertex_pipeline_job = PipelineJob(
    display_name=f"gemini-tuning-pipeline-{date}",
    template_path=PIPELINE_PACKAGE_PATH,
    pipeline_root=pipeline_root,
    parameter_values={
        **pipeline_arguments,
        "pipeline_root": pipeline_root,
        "training_source_gcs_uri": training_source_gcs_uri,
        "evaluation_source_gcs_uri": evaluation_source_gcs_uri,
    },
    location=REGION,
)

print(vertex_pipeline_job)


Submitting pipeline as: vertex-pipeline-runner@project-2bb7b579-5b60-4790-8c2.iam.gserviceaccount.com


Failed to convert project number to project ID.
Traceback (most recent call last):
  File "c:\llmops-first-demo\.llmops_first_demo\Lib\site-packages\google\api_core\grpc_helpers.py", line 55, in error_remapped_callable
    return callable_(*args, **kwargs)
  File "c:\llmops-first-demo\.llmops_first_demo\Lib\site-packages\grpc\_interceptor.py", line 276, in __call__
    response, ignored_call = self._with_call(
                             ~~~~~~~~~~~~~~~^
        request,
        ^^^^^^^^
    ...<4 lines>...
        compression=compression,
        ^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\llmops-first-demo\.llmops_first_demo\Lib\site-packages\grpc\_interceptor.py", line 331, in _with_call
    return call.result(), call
           ~~~~~~~~~~~^^
  File "c:\llmops-first-demo\.llmops_first_demo\Lib\site-packages\grpc\_channel.py", line 438, in result
    raise self
  File "c:\llmops-first-demo\.llmops_first_demo\Lib\site-packages\grpc\_interceptor.py", line 314, in continuation
    

In [17]:
vertex_pipeline_job.submit(service_account=PIPELINE_RUNNER_SERVICE_ACCOUNT)
print(vertex_pipeline_job.resource_name)


Creating PipelineJob
PipelineJob created. Resource name: projects/838831985868/locations/us-central1/pipelineJobs/gemini-tuning-pipeline-20260419103935
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/838831985868/locations/us-central1/pipelineJobs/gemini-tuning-pipeline-20260419103935')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/gemini-tuning-pipeline-20260419103935?project=838831985868
projects/838831985868/locations/us-central1/pipelineJobs/gemini-tuning-pipeline-20260419103935


In [18]:
from google.cloud.aiplatform import PipelineJob

latest_pipeline_job = PipelineJob.get(
    resource_name=vertex_pipeline_job.resource_name,
    project=PROJECT_ID,
    location=REGION,
    credentials=credentials,
)
print(latest_pipeline_job.state)


3
